# Pure DDPG Portfolio Workflow

This notebook is the replacement for `main_code.ipynb`.

It keeps the same core idea, but the code is split into short workflow cells.


## 0. Import shared workflow API

- Put reusable functions in `src/finance_rl_slm/`.
- Use the same helper API as the script version.


In [ ]:
from __future__ import annotations

import sys
from dataclasses import replace
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "main":
    PROJECT_ROOT = PROJECT_ROOT.parent

# add the sys.path to this file
for path in (PROJECT_ROOT / "src", PROJECT_ROOT, PROJECT_ROOT / "src" / "FinRL"):
    path_str = str(path) # The data path transformer
    if path.exists() and path_str not in sys.path:
        sys.path.insert(0, path_str)

from finance_rl_slm.config import DEFAULT_CONFIG # The dataclass for date
from finance_rl_slm.workflow import (
    build_daily_sentiment,
    build_sentiment_inputs,
    load_online_price_data,
    load_price_data,
    plot_normalized_prices,
    print_runtime_context,
    result_picture_path,
    result_profile_path,
    run_only_ddpg_online,
    split_price_data,
)
from finance_rl_slm.training import train_offline_model


## 1. Configure the experiment

- Online test window: `2026-01-01` to `2026-06-21`.
- Pipeline name: `only_ddpg`.
- Sentiment input: disabled.


In [ ]:
# The date replace

config = replace(
    DEFAULT_CONFIG,
    online_start="2026-01-01",
    online_end="2026-06-21",
)
print_runtime_context(config)


## 2. Run online evaluation without SLM

- This is the default notebook path.
- It loads the existing `ddpg_portfolio_offline.zip` model.
- It does not train a new model before online testing.


In [ ]:
only_ddpg_logs = run_only_ddpg_online(config)
only_ddpg_logs.head()


## 3. Optional offline training

- Keep this cell disabled during normal online testing.
- Turn it on only when you need to rebuild `ddpg_portfolio_offline.zip`.
- `split_price_data()` checks train and validation ranges before training.


In [ ]:
RUN_OPTIONAL_TRAINING = False

if RUN_OPTIONAL_TRAINING:
    price_df = load_price_data(config)
    plot_normalized_prices(price_df, config)

    splits = split_price_data(price_df, config)
    model = train_offline_model(splits.train, splits.valid, config)
else:
    print("Optional training skipped. Set RUN_OPTIONAL_TRAINING = True to retrain.")


## 4. Output notes

- Figure files:
  - `addenda/result_picture/online_reward_only_ddpg_2026-01-01_2026-06-21.png`
  - `addenda/result_picture/online_wealth_only_ddpg_2026-01-01_2026-06-21.png`
  - `addenda/result_picture/online_daily_return_only_ddpg_2026-01-01_2026-06-21.png`

- Profile file:
  - `addenda/result_profile_comparse/only_ddpg_online_profile_2026-01-01_2026-06-21.csv`

- Use this profile as the baseline when comparing against DDPG+SLM.


## Workflow Notes - Pure DDPG

- The main agent is a price-based DDPG policy.

- The training environment uses:
  - recent portfolio returns,
  - MACD-like wealth trend,
  - Bollinger-style wealth deviation,
  - normalized wealth.

- The online evaluation uses the same trained model.

- No language model is used in this pipeline.

- The goal is to understand the baseline behavior first:
  - Does wealth grow or shrink?
  - Is reward stable or noisy?
  - How large is the drawdown?

- After this notebook runs, run the SLM notebook and compare the two profile CSV files.
